In [123]:
import os
import glob
from datasets import Dataset, concatenate_datasets

### Load All documents from `docs`

In [126]:
all_docs = glob.glob(os.path.join('../../../docs/', '**', '*.md'), recursive=True)
document_outline_template = """
sdg_hub is a python package for synthetic data generation using a large language model to train another LLM. 
"""
doc_code = []
doc_outline = []
for doc in all_docs:
    print(doc)
    with open(doc, 'r') as f:
        doc_code.append(f.read())
    doc_outline.append(document_outline_template)

../../../docs/structure.md
../../../docs/blocks/block_init.md
../../../docs/blocks/utilblocks.md
../../../docs/blocks/llmblock_class.md
../../../docs/blocks/block_class.md
../../../docs/examples/data-generation-with-llama-70b/data-generation-with-llama-70b.md
../../../docs/examples/knowledge_tuning/instructlab/knowledge/document_pre_processing.md
../../../docs/examples/knowledge_tuning/instructlab/knowledge/knowledge_generation_and_mixing.md
../../../docs/examples/knowledge_tuning/instructlab/knowledge/document_collection/ibm-annual-report/ibm-annual-report-2024.md
../../../docs/examples/knowledge_tuning/knowledge_tuning_with_reasoning_model/reasoning_sdg_quality.md
../../../docs/examples/knowledge_tuning/knowledge_tuning_with_reasoning_model/README.md
../../../docs/examples/knowledge_tuning/knowledge_tuning_with_reasoning_model/reasoning_sdg_financebench.md
../../../docs/examples/instructlab/skills/mdtable_manipulation.md
../../../docs/examples/instructlab/skills/unstructured_to_mdtab

### ICL

In [127]:
icls_knowledge = {
  "icl_document": '# Documentation: src/sdg_hub/blocks/block.py - Base Block Class\n\nThis module defines the abstract base class (`Block`) for all computational units within the Synthetic Data Generation (SDG) Hub framework. It provides core functionalities common to all blocks, such as configuration loading and input validation, particularly for blocks that utilize templating.\n\n---\n\n## Module Overview\n\nThe `block.py` module is central to the `sdg_hub`\'s architecture. It ensures that all specialized blocks adhere to a common interface and inherit baseline functionalities.\n\n* **SPDX-License-Identifier**: `Apache-2.0` (Indicates the licensing terms for this file).\n\n---\n\n## Imports\n\nThe module utilizes several standard Python libraries, third-party libraries, and local project modules:\n\n* **Standard Library**:\n    * `abc.ABC`: Used to declare abstract base classes.\n    * `collections.ChainMap`: To combine multiple dictionaries for template rendering, allowing defaults or overrides.\n    * `typing.Any, Dict, Optional`: For type hinting.\n* **Third-Party**:\n    * `jinja2.Template, UndefinedError`: For handling Jinja2 templating and associated errors.\n    * `yaml`: For parsing YAML configuration files.\n* **Local Project Modules**:\n    * `..registry.BlockRegistry`: Imports the `BlockRegistry` to allow block classes to register themselves, making them discoverable by the framework.\n    * `..logger_config.setup_logger`: Imports a function to set up a structured logger for the module.\n\n---\n\n## Logger Configuration\n\nA logger instance is initialized for this module to facilitate consistent logging practices:\n\n```python\nlogger = setup_logger(__name__)\n```\n\n---\n\n## The `Block` Class\n\nThe `Block` class is the cornerstone of the framework\'s modular design.\n\n```python\n@BlockRegistry.register("Block")\nclass Block(ABC):\n    """Base abstract class for all blocks in the system.\n\n    This class provides common functionality for block validation and configuration loading.\n    All specific block implementations should inherit from this class.\n    """\n\n    def __init__(self, block_name: str) -> None:\n        self.block_name = block_name\n\n    @staticmethod\n    def _validate(prompt_template: Template, input_dict: Dict[str, Any]) -> bool:\n        # ... (implementation details below)\n        pass\n\n    def _load_config(self, config_path: str) -> Optional[Dict[str, Any]]:\n        # ... (implementation details below)\n        pass\n```',
  "icl_query_1": "What is the role of `block.py` in the sdg_hub framework?",
  "icl_response_1": "The `block.py` module defines the abstract base class `Block` that all other blocks in the SDG Hub framework must inherit from. It provides shared core functionalities such as configuration loading and template-based input validation. This ensures consistency and modularity across all computational units in the system.",
  "icl_query_2": "How does the `Block` class support templating and configuration in sdg_hub?",
  "icl_response_2": "The `Block` class includes a static method `_validate` that uses Jinja2 templates to validate input dictionaries. It also has a method `_load_config` for reading configuration files in YAML format. These features help standardize how each block processes input and configuration across the framework.",
  "icl_query_3": "What does `@BlockRegistry.register(\"Block\")` do in the block.py file?",
  "icl_response_3": "This decorator registers the `Block` class with the `BlockRegistry`, making it discoverable by the SDG Hub framework. It enables the dynamic lookup and instantiation of blocks by name, supporting plug-and-play extensibility across different components of the system.",
}

icl_code = {
  "icl_document": '\nsdg_hub is a package for synthetic data generation using a large language model to train another LLM. \nFile Path: ../../../src/sdg_hub/blocks/llmblock.py\nPackage Path: sdg_hub.blocks.llmblock\n\n```python\n# SPDX-License-Identifier: Apache-2.0\n"""Utility blocks for dataset manipulation and transformation.\n\nThis module provides various utility blocks for operations like column manipulation,\ndata population, selection, and transformation of datasets. These blocks are designed\nto work with the Hugging Face datasets library and provide common data processing\nfunctionality.\n\nThe module includes blocks for:\n\nData Filtering and Selection:\n- "FilterByValueBlock": Filter datasets based on column values with support for:\n  * Multiple comparison operations (equals, contains, etc.)\n  * Data type conversion\n  * Batch processing\n  * Custom filtering conditions\n\nData Population and Configuration:\n- "SamplePopulatorBlock": Populate datasets with configuration data:\n  * Load data from multiple YAML configuration files\n  * Map configuration data to dataset columns\n  * Support for post-fix naming\n  * Batch processing capabilities\n\nColumn Operations:\n- "SelectorBlock": Map values between columns based on choice mapping\n- "CombineColumnsBlock": Concatenate multiple columns with custom separators\n- "FlattenColumnsBlock": Transform wide format to long format datasets\n- "DuplicateColumns": Create copies of existing columns with new names\n- "RenameColumns": Rename columns using a mapping dictionary\n- "SetToMajorityValue": Set column values to the most frequent value\n\nAdvanced Processing:\n- "IterBlock": Apply blocks iteratively for multiple generations:\n  * Support for any block type\n  * Configurable number of iterations\n  * Custom generation parameters\n  * Batch processing support\n\nAll blocks support:\n- Batch processing for improved performance\n- Error handling and logging\n- Type hints and comprehensive documentation\n- Integration with the BlockRegistry\n- Standardized interface through the Block base class\n"""\n\n# Standard\nimport operator\nfrom typing import Any, Callable, Dict, List, Optional, Type, Union\n\n# Third Party\nfrom datasets import Dataset\n\n# Local\nfrom .block import Block\nfrom ..registry import BlockRegistry\nfrom ..logger_config import setup_logger\n\nlogger = setup_logger(__name__)\n\n\n@BlockRegistry.register("FilterByValueBlock")\nclass FilterByValueBlock(Block):\n    """A block for filtering datasets based on column values.\n\n    This block allows filtering of datasets using various operations (e.g., equals, contains)\n    on specified column values, with optional data type conversion. It supports both\n    single value and list of values for filtering.\n\n    Parameters\n    ----------\n    block_name : str\n        Name of the block.\n    filter_column : str\n        The name of the column in the dataset to apply the filter on.\n    filter_value : Union[Any, List[Any]]\n        The value(s) to filter by. Can be a single value or a list of values.\n    operation : Callable[[Any, Any], bool]\n        A binary operator from the operator module (e.g., operator.eq, operator.contains)\n        that takes two arguments and returns a boolean.\n    convert_dtype : Optional[Union[Type[float], Type[int]]], optional\n        Type to convert the filter column to. Can be either float or int.\n        If None, no conversion is performed.\n    **batch_kwargs : Dict[str, Any]\n        Additional kwargs for batch processing.\n    """\n\n    def __init__(\n        self,\n        block_name: str,\n        filter_column: str,\n        filter_value: Union[Any, List[Any]],\n        operation: Callable[[Any, Any], bool],\n        convert_dtype: Optional[Union[Type[float], Type[int]]] = None,\n        **batch_kwargs: Dict[str, Any],\n    ) -> None:\n        """Initialize a new FilterByValueBlock instance.\n\n        Parameters\n        ----------\n        block_name : str\n            Name of the block.\n        filter_column : str\n            The name of the column in the dataset to apply the filter on.\n        filter_value : Union[Any, List[Any]]\n            The value(s) to filter by. Can be a single value or a list of values.\n        operation : Callable[[Any, Any], bool]\n            A binary operator from the operator module (e.g., operator.eq, operator.contains)\n            that takes two arguments and returns a boolean.\n        convert_dtype : Optional[Union[Type[float], Type[int]]], optional\n            Type to convert the filter column to. Can be either float or int.\n            If None, no conversion is performed.\n        **batch_kwargs : Dict[str, Any]\n            Additional kwargs for batch processing.\n\n        Returns\n        -------\n        None\n\n        Raises\n        ------\n        ValueError\n            If the operation is not from the operator module.\n        """\n        super().__init__(block_name=block_name)\n        # Validate that operation is from operator module\n        if operation.__module__ != "_operator":\n            logger.error("Invalid operation: %s", operation)\n            raise ValueError("Operation must be from operator module")\n\n        self.value = filter_value if isinstance(filter_value, list) else [filter_value]\n        self.column_name = filter_column\n        self.operation = operation\n        self.convert_dtype = convert_dtype\n        self.num_procs = batch_kwargs.get("num_procs", 1)\n\n    def _convert_dtype(self, sample: Dict[str, Any]) -> Dict[str, Any]:\n        """Convert the data type of the filter column.\n\n        Parameters\n        ----------\n        sample : Dict[str, Any]\n            The sample dictionary containing the column to convert.\n\n        Returns\n        -------\n        Dict[str, Any]\n            The sample with converted column value.\n        """\n        try:\n            sample[self.column_name] = self.convert_dtype(sample[self.column_name])\n        except ValueError as e:\n            logger.error(\n                "Error converting dtype: %s, filling with None to be filtered later", e\n            )\n            sample[self.column_name] = None\n        return sample\n\n    def generate(self, samples: Dataset) -> Dataset:\n        """Generate filtered dataset based on specified conditions.\n\n        Parameters\n        ----------\n        samples : Dataset\n            The input dataset to filter.\n\n        Returns\n        -------\n        Dataset\n            The filtered dataset.\n        """\n        if self.convert_dtype:\n            samples = samples.map(\n                self._convert_dtype,\n                num_proc=self.num_procs,\n            )\n\n        if self.operation == operator.contains:\n            samples = samples.filter(\n                lambda x: self.operation(self.value, x[self.column_name]),\n                num_proc=self.num_procs,\n            )\n\n        samples = samples.filter(\n            lambda x: x[self.column_name] is not None,\n            num_proc=self.num_procs,\n        )\n\n        samples = samples.filter(\n            lambda x: any(\n                self.operation(x[self.column_name], value) for value in self.value\n            ),\n            num_proc=self.num_procs,\n        )\n\n        return samples',
  "icl_query_1": "What is the purpose of the FilterByValueBlock in the sdg_hub package?",
  "icl_response_1": "The FilterByValueBlock is designed to filter datasets based on values in a specific column using comparison operations like equals or contains. It supports single or multiple values, optional data type conversion, and batch processing, and is integrated with the Hugging Face datasets library.",
  "icl_query_2": "How does FilterByValueBlock ensure valid operations for filtering?",
  "icl_response_2": "During initialization, the block checks if the provided operation (like operator.eq or operator.contains) comes from Python's `operator` module. If not, it logs an error and raises a ValueError to prevent invalid filtering behavior.",
  "icl_query_3": "What happens if data type conversion fails in FilterByValueBlock?",
  "icl_response_3": "If the `_convert_dtype` method fails to convert a value (e.g., due to a ValueError), the method logs an error and sets the column value to `None`. These `None` values are later filtered out during dataset processing to maintain clean and valid results.",
}

### Load All code files

In [128]:
base_dir = "../../../src/sdg_hub"

blocks = os.path.join(base_dir, "blocks")

configs = os.path.join(base_dir, "configs")

flows = os.path.join(base_dir, "flows")


misc_files = [os.path.join(base_dir, "checkpointer.py") , os.path.join(base_dir, "flow.py"), os.path.join(base_dir, "pipeline.py"), os.path.join(base_dir, "prompts.py"), os.path.join(base_dir, "registry.py"), os.path.join(base_dir, "sdg.py"),
os.path.join(base_dir, "utils", "datautils.py")]

misc_group =  {
    'core_componenets': [os.path.join(base_dir, "flow.py"), os.path.join(base_dir, "pipeline.py"), os.path.join(base_dir, "prompts.py"), os.path.join(base_dir, "registry.py"), os.path.join(base_dir, "sdg.py")],
    'utils': [os.path.join(base_dir, "utils", "datautils.py")],
    'aux_componenets': [os.path.join(base_dir, "checkpointer.py")]
}

document_outline_template = """
sdg_hub is a python package for synthetic data generation using a large language model to train another LLM. 
File Path: {file_path}
Package Path: {package_path}
"""

document_outline_prompts = """
sdg_hub is a python package for synthetic data generation using a large language model to train another LLM. 
Prompt Path: {file_path}
Prompt Taxonomy: {prompt_tax}
"""

document_outline_flows = """
sdg_hub is a python package for synthetic data generation using a large language model to train another LLM. 
YAML Flow Path: {file_path}
Flow Taxonomy: {flow_tax}
"""

In [129]:
block_files = [os.path.join(blocks, f) for f in os.listdir(blocks) if 'init' not in f and 'pycache' not in f]
block_code = []
block_doc_outline = []
for block_file in block_files:
    with open(block_file, 'r') as f:
        block_code.append(f"```python\n{f.read()}\n```")
    package_path = block_file[block_file.find('src/'):].replace('src/', '').replace('.py', '').split('/')
    package_path = '.'.join(package_path)
    block_doc_outline.append(document_outline_template.format(file_path=block_file, package_path=package_path))

In [130]:
config_files =  glob.glob(os.path.join(configs, "*", "*.yaml"))
config_file_code = []
config_file_outline = []
for config_file in config_files:
    with open(config_file, 'r') as f:
        config_file_code.append(f"```yaml\n{f.read()}\n```")
    file_path = config_file[config_file.find('src/'):].replace('src/', '')
    prompt_tax = file_path.replace('sdg_hub/configs/', '').split('/')
    prompt_tax = f"Type : {prompt_tax[0]} \nPrompt: {prompt_tax[1].replace('.yaml', '')}"
    config_file_outline.append(document_outline_prompts.format(file_path=file_path, prompt_tax=prompt_tax))


In [131]:
flow_files = glob.glob(os.path.join(flows, "**", "*.yaml"), recursive=True)
flow_file_code = []
flow_file_outline = []
for flow_file in flow_files:
    with open(flow_file, 'r') as f:
        flow_file_code.append(f"```yaml\n{f.read()}\n```")
    file_path = flow_file[flow_file.find('src/'):].replace('src/', '')
    flow_tax = file_path.replace('sdg_hub/flows/', '').split('/')
    flow_tax = ' -> '.join([e.replace('.yaml', '') for e in flow_tax])
    flow_file_outline.append(document_outline_flows.format(file_path=file_path, flow_tax=flow_tax))


In [132]:
misc_file_code = []
misc_file_outline = []
for misc_file in misc_files:
    with open(misc_file, 'r') as f:
        misc_file_code.append(f"```python\n{f.read()}\n```")
    file_path = misc_file[misc_file.find('src/'):].replace('src/', '')
    package_path = '.'.join(file_path.replace('.py', '').split('/'))
    misc_file_outline.append(document_outline_template.format(file_path=file_path, package_path=package_path))

In [133]:
final_data =  block_code + config_file_code + flow_file_code + misc_file_code
final_data_doc_outline =  block_doc_outline + config_file_outline + flow_file_outline + misc_file_outline
ds_code = Dataset.from_dict({'document_outline': final_data_doc_outline, 'document': final_data})
ds_code = ds_code.map(lambda x: icl_code)

ds_doc = Dataset.from_dict({'document_outline': doc_outline, 'document': doc_code})
ds_doc = ds_doc.map(lambda x: icls_knowledge)

ds = concatenate_datasets([ds_doc, ds_code])

Map: 100%|██████████| 15/15 [00:00<00:00, 5195.69 examples/s]


In [135]:
ds.to_json("seed_data.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 61.44ba/s]


1971832